In [ ]:
import os
import re
import time
import json
import numpy as np
import pandas as pd
from collections import defaultdict, Counter
from pathlib import Path

from Bio import Entrez, SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

# --- CONFIGURE THESE ---
Entrez.email = None # Required
Entrez.api_key = None # Optional
DATA_DIR = None # Required
DATA_DIR.mkdir(exist_ok=True)

WINDOW_UPSTREAM = 50       # nt upstream of slippage site
WINDOW_DOWNSTREAM = 200    # nt downstream of slippage site
WINDOW_SIZE = WINDOW_UPSTREAM + WINDOW_DOWNSTREAM  # 250 nt total

NEG_RATIO_DECOY = 4        # decoy negatives per positive
NEG_RATIO_RANDOM = 2       # random CDS negatives per positive

In [ ]:
# ============================================================
# CELL 2 — Download & Curate PRF Records from NCBI
# ============================================================
#
# Strategy:
#   1. Download all viral RefSeq records with "ribosomal slippage" (~1,500)
#   2. Fetch known cellular frameshifted genes by targeted queries (~50)
#   3. Verify landmark PRF sites are present
#
# Targets: ~10,000 positives (after augmentation), ~100,000 negatives
# ============================================================

import time
from pathlib import Path
from Bio import Entrez, SeqIO

# ---- Query definitions ----

QUERIES = {
    "viral_refseq": (
        '"ribosomal slippage"[All Fields] '
        'AND Viruses[Organism] '
        'AND srcdb_refseq[PROP]'
    ),
    "viral_all": (
        '"ribosomal slippage"[All Fields] '
        'AND Viruses[Organism]'
    ),
    "targeted_families": (
        '("ribosomal slippage"[All Fields] OR "ribosomal frameshift"[All Fields]) '
        'AND (Retroviridae[Organism] OR Coronaviridae[Organism] '
        'OR Totiviridae[Organism] OR Luteoviridae[Organism] '
        'OR Astroviridae[Organism] OR Tombusviridae[Organism] '
        'OR Arteriviridae[Organism] OR Paramyxoviridae[Organism] '
        'OR Flaviviridae[Organism] OR Reoviridae[Organism] '
        'OR Birnaviridae[Organism] OR Partitiviridae[Organism] '
        'OR Chrysoviridae[Organism] OR Potyviridae[Organism] '
        'OR Narnaviridae[Organism] OR Hypoviridae[Organism]) '
        'AND srcdb_refseq[PROP]'
    ),
}

# Known cellular genes with programmed ribosomal frameshifting.
# Format: gene_name -> (NCBI organism filter, PRF direction, description)
KNOWN_CELLULAR_PRF_GENES = {
    # +1 PRF — Antizyme family (polyamine-regulated, all eukaryotes)
    "OAZ1":    ("Mammalia[Organism]",                   +1, "Ornithine decarboxylase antizyme 1"),
    "OAZ2":    ("Mammalia[Organism]",                   +1, "Ornithine decarboxylase antizyme 2"),
    "OAZ3":    ("Mammalia[Organism]",                   +1, "Ornithine decarboxylase antizyme 3"),
    "oaz1":    ("Danio rerio[Organism]",                +1, "Antizyme 1 — zebrafish"),
    "oaz2":    ("Danio rerio[Organism]",                +1, "Antizyme 2 — zebrafish"),
    "antizyme":("Drosophila melanogaster[Organism]",    +1, "Antizyme — fruit fly"),

    # +1 PRF — Bacterial release factor
    "prfB":    ("Bacteria[Organism]",                   +1, "Release factor 2"),

    # -1 PRF — Bacterial
    "dnaX":    ("Bacteria[Organism]",                   -1, "DNA polymerase III tau/gamma subunit"),
    "copA":    ("Bacteria[Organism]",                   -1, "Copper transporter ATPase"),
    "cdd":     ("Bacteria[Organism]",                   -1, "CTP deaminase"),

    # +1 PRF — Yeast
    "ABP140":  ("Saccharomyces cerevisiae[Organism]",   +1, "Actin-binding protein 140"),
    "EST3":    ("Saccharomyces cerevisiae[Organism]",   +1, "Telomerase subunit EST3"),
    "EST1":    ("Saccharomyces cerevisiae[Organism]",   +1, "Telomerase subunit EST1"),
    "TY1":     ("Saccharomyces cerevisiae[Organism]",   +1, "Ty1 retrotransposon gag-pol"),
    "TY3":     ("Saccharomyces cerevisiae[Organism]",   +1, "Ty3 retrotransposon gag-pol"),

    # -1 PRF — Mammalian / Human cellular genes
    "MA3":     ("Homo sapiens[Organism]",               -1, "Paraneoplastic antigen Ma3"),
    "PEG10":   ("Mammalia[Organism]",                   -1, "Paternally expressed gene 10"),
    "CCR5":    ("Homo sapiens[Organism]",               -1, "C-C chemokine receptor type 5"),
    "Edr":     ("Mammalia[Organism]",                   -1, "Mammalian -1 PRF gene Edr"),
    "APC":     ("Homo sapiens[Organism]",               -1, "Adenomatous polyposis coli"),
    "LIPA":    ("Homo sapiens[Organism]",               -1, "Lysosomal acid lipase"),

    # Endogenous retroviruses (borderline viral/cellular)
    "HERVK":   ("Homo sapiens[Organism]",               -1, "Human endogenous retrovirus K"),
}


# ---- Helper functions ----

def search_ncbi(query, label=""):
    """Search NCBI and return count + history server refs."""
    handle = Entrez.esearch(
        db="nucleotide",
        term=query,
        retmax=0,
        usehistory="y"
    )
    result = Entrez.read(handle)
    handle.close()
    count = int(result["Count"])
    if label:
        print(f"  {label:30s} → {count:>8,} records")
    return count, result["WebEnv"], result["QueryKey"]


def download_genbank_batches(count, webenv, query_key, batch_size=200, label=""):
    """
    Download GenBank records in batches via Entrez history server.
    Saves raw .gb files to DATA_DIR/genbank_raw/.
    Returns the output directory path.
    """
    outdir = DATA_DIR / "genbank_raw"
    outdir.mkdir(exist_ok=True, parents=True)

    total_downloaded = 0
    batch_num = 0
    # Use label in filename to avoid overwriting between queries
    prefix = label.replace(" ", "_") if label else "batch"

    for start in range(0, count, batch_size):
        batch_num += 1
        end = min(start + batch_size, count)
        print(f"    [{label}] batch {batch_num}: records {start+1}-{end} of {count}", end="")

        for attempt in range(3):
            try:
                handle = Entrez.efetch(
                    db="nucleotide",
                    rettype="gb",
                    retmode="text",
                    retstart=start,
                    retmax=batch_size,
                    webenv=webenv,
                    query_key=query_key,
                )
                data = handle.read()
                handle.close()
                break
            except Exception as e:
                print(f" (retry {attempt+1}/3: {e})", end="")
                time.sleep(5)
        else:
            print(" FAILED — skipping batch")
            continue

        outfile = outdir / f"{prefix}_{batch_num:04d}.gb"
        with open(outfile, "w") as f:
            f.write(data)

        total_downloaded += (end - start)
        print(f" ✓")
        # Rate limit: 3/sec without API key, 10/sec with
        time.sleep(0.4 if Entrez.api_key else 1.0)

    print(f"    → Downloaded {total_downloaded} records\n")
    return outdir


def fetch_cellular_prf_genes():
    """
    Fetch known cellular frameshifted genes by targeted per-gene queries.
    Much more precise than the broad 1.1M-result text search.
    Returns count of genes successfully fetched.
    """
    outdir = DATA_DIR / "genbank_raw"
    outdir.mkdir(exist_ok=True, parents=True)

    fetched = 0
    failed = []

    for gene, (org_filter, direction, desc) in KNOWN_CELLULAR_PRF_GENES.items():
        query = f'{gene}[Gene] AND {org_filter} AND srcdb_refseq[PROP]'

        try:
            handle = Entrez.esearch(db="nucleotide", term=query, retmax=10)
            result = Entrez.read(handle)
            handle.close()

            ids = result["IdList"]
            if not ids:
                # Try broader search without RefSeq filter
                query_broad = f'{gene}[Gene] AND {org_filter}'
                handle = Entrez.esearch(db="nucleotide", term=query_broad, retmax=5)
                result = Entrez.read(handle)
                handle.close()
                ids = result["IdList"]

            if not ids:
                failed.append(gene)
                print(f"    {gene:12s} ({desc:45s}) — ✗ no results")
                continue

            # Fetch top records (cap at 5 per gene to avoid redundancy)
            fetch_ids = ids[:5]
            handle = Entrez.efetch(
                db="nucleotide",
                id=",".join(fetch_ids),
                rettype="gb",
                retmode="text",
            )
            data = handle.read()
            handle.close()

            outfile = outdir / f"cellular_{gene}.gb"
            with open(outfile, "w") as f:
                f.write(data)

            fetched += 1
            print(f"    {gene:12s} ({desc:45s}) — ✓ {len(fetch_ids)} records")
            time.sleep(0.4 if Entrez.api_key else 1.0)

        except Exception as e:
            failed.append(gene)
            print(f"    {gene:12s} ({desc:45s}) — ✗ ERROR: {e}")
            continue

    print(f"\n    Fetched: {fetched}/{len(KNOWN_CELLULAR_PRF_GENES)} genes")
    if failed:
        print(f"    Failed:  {', '.join(failed)}")

    return fetched


def verify_landmark_sites(positives):
    """
    After parsing positives, check that well-known PRF sites are
    represented. Prints warnings for anything missing so you can
    add them manually if needed.
    """
    landmarks = {
        "HIV-1 gag-pol (-1)":        ["hiv", "gag", "pol"],
        "SARS-CoV ORF1a/1b (-1)":    ["sars", "coronavirus", "orf1a", "1a-1b", "replicase"],
        "RSV gag-pol (-1)":          ["rous sarcoma", "rsv"],
        "HTLV gag-pro (-1)":         ["htlv", "human t-cell", "t-lymphotropic"],
        "Antizyme / OAZ (+1)":       ["antizyme", "oaz", "ornithine decarboxylase"],
        "E.coli dnaX (-1)":          ["dnax", "dna polymerase iii"],
        "E.coli prfB / RF2 (+1)":    ["prfb", "release factor 2", "peptide chain release"],
        "Yeast Ty1/Ty3 (+1)":        ["ty1", "ty3", "retrotransposon"],
        "PEG10 (-1)":                ["peg10", "paternally expressed"],
        "Coronavirus IBV (-1)":      ["infectious bronchitis", "ibv", "avian coronavirus"],
    }

    print("\n" + "=" * 60)
    print("LANDMARK PRF SITE VERIFICATION")
    print("=" * 60)

    all_text = []
    for p in positives:
        text = " ".join([
            str(p.get("gene", "")),
            str(p.get("product", "")),
            str(p.get("organism", "")),
        ]).lower()
        all_text.append(text)

    found_count = 0
    missing = []
    for name, keywords in landmarks.items():
        found = any(
            any(kw in text for kw in keywords)
            for text in all_text
        )
        if found:
            found_count += 1
            print(f"  ✓  {name}")
        else:
            missing.append(name)
            print(f"  ✗  {name}  ← MISSING")

    print(f"\nFound {found_count}/{len(landmarks)} landmark sites")
    if missing:
        print(f"Missing sites may need manual addition or different search terms.")
        print(f"This is OK for a hackathon — viral records should cover most cases.")
    print("=" * 60)


# ---- Main execution ----

print("=" * 60)
print("STEP 1: Check query sizes")
print("=" * 60)

query_counts = {}
for name, query in QUERIES.items():
    count, _, _ = search_ncbi(query, label=name)
    query_counts[name] = count
    time.sleep(0.5)


print("\n" + "=" * 60)
print("STEP 2: Download viral RefSeq records")
print("=" * 60)

count, webenv, query_key = search_ncbi(QUERIES["viral_refseq"])
genbank_dir = download_genbank_batches(
    count, webenv, query_key,
    batch_size=200,
    label="viral_refseq"
)

# If viral_refseq gives < 1000 unique positives after parsing,
# uncomment the following to also pull non-RefSeq viral records:
#
# SUPPLEMENT_WITH_ALL_VIRAL = False
# if SUPPLEMENT_WITH_ALL_VIRAL:
#     print("Supplementing with non-RefSeq viral records...")
#     count2, webenv2, qk2 = search_ncbi(QUERIES["viral_all"])
#     # Cap at 10,000 to keep it manageable
#     download_genbank_batches(
#         min(count2, 10000), webenv2, qk2,
#         batch_size=200,
#         label="viral_all"
#     )


print("=" * 60)
print("STEP 3: Fetch known cellular PRF genes")
print("=" * 60)

cellular_count = fetch_cellular_prf_genes()


print("\n" + "=" * 60)
print("STEP 4: Summary")
print("=" * 60)

gb_files = list((DATA_DIR / "genbank_raw").glob("*.gb"))
total_file_size = sum(f.stat().st_size for f in gb_files) / (1024 * 1024)

print(f"  GenBank files downloaded:  {len(gb_files)}")
print(f"  Total file size:           {total_file_size:.1f} MB")
print(f"  Viral RefSeq records:      ~{query_counts['viral_refseq']}")
print(f"  Cellular genes fetched:    {cellular_count}")
print()
print("  Expected yield after parsing:")
print(f"    Raw positives:           ~1,000 - 2,500")


# Store genbank_dir for use in subsequent cells
# (already set above, but explicit for clarity)
genbank_dir = DATA_DIR / "genbank_raw"

In [ ]:
genbank_dir = DATA_DIR / "genbank_raw"
def parse_slippage_sites(genbank_dir):
    """
    Parse all downloaded GenBank files and extract:
      - The full genome/sequence
      - CDS features with /ribosomal_slippage qualifier
      - The frameshift position (from join coordinates)
      - Frameshift direction (-1 or +1)

    Returns a list of dicts with extracted info.
    """
    sites = []

    gb_files = sorted(Path(genbank_dir).glob("*.gb"))
    print(f"Parsing {len(gb_files)} GenBank batch files...")

    skipped_undefined = 0

    for gb_file in gb_files:
        try:
            records = list(SeqIO.parse(str(gb_file), "genbank"))
        except Exception as e:
            print(f"  Error parsing {gb_file.name}: {e}")
            continue

        for record in records:
            try:
                seq_str = str(record.seq).upper()
            except Exception:
                skipped_undefined += 1
                continue
            accession = record.id
            organism = ""
            taxonomy = []

            # Extract organism and taxonomy from annotations
            if "organism" in record.annotations:
                organism = record.annotations["organism"]
            if "taxonomy" in record.annotations:
                taxonomy = record.annotations["taxonomy"]

            for feature in record.features:
                if feature.type != "CDS":
                    continue

                # Check for ribosomal_slippage qualifier
                qualifiers = feature.qualifiers
                has_slippage = "ribosomal_slippage" in qualifiers

                if not has_slippage:
                    continue

                # Extract the frameshift position from the location
                # Frameshifted CDS have compound locations like:
                #   join(300..800, 799..1500)  → -1 frameshift at 800
                #   join(300..800, 802..1500)  → +1 frameshift at 800
                location = feature.location
                if not hasattr(location, 'parts') or len(location.parts) < 2:
                    continue

                parts = location.parts

                for i in range(len(parts) - 1):
                    part_a = parts[i]
                    part_b = parts[i + 1]

                    end_a = int(part_a.end)      # end of first exon
                    start_b = int(part_b.start)  # start of second exon

                    # Determine frameshift direction from the gap/overlap
                    gap = start_b - end_a
                    # gap = -1 means 1nt overlap → -1 frameshift
                    # gap = +1 means 1nt gap    → +1 frameshift
                    # gap = 0 could be ambiguous

                    if gap == -1:
                        direction = -1
                    elif gap == 1:
                        direction = +1
                    elif gap == 0:
                        direction = -1  # ambiguous, default to -1
                    else:
                        # Unusual gap, might be splicing or something else
                        direction = gap

                    slippage_pos = end_a  # position of the frameshift

                    # Extract the gene name if available
                    gene = qualifiers.get("gene", ["unknown"])[0]
                    product = qualifiers.get("product", ["unknown"])[0]

                    sites.append({
                        "accession": accession,
                        "organism": organism,
                        "taxonomy": "|".join(taxonomy),
                        "gene": gene,
                        "product": product,
                        "slippage_pos": slippage_pos,
                        "direction": direction,
                        "seq_length": len(seq_str),
                        "strand": int(part_a.strand) if part_a.strand else 1,
                        "sequence": seq_str,
                    })

    print(f"Extracted {len(sites)} slippage sites from {len(gb_files)} files")
    print(f"Skipped {skipped_undefined} records with undefined sequences (WGS master records)")
    return sites


sites = parse_slippage_sites(genbank_dir)
sites_df = pd.DataFrame(sites)
# Real PRF events shift by -1, +1, or occasionally -2, +2
# Anything else is an intron, segmented gene, or annotation artifact
VALID_DIRECTIONS = {-1, +1, -2, +2}

print(f"Before filtering: {len(sites_df)} sites")

sites_df = sites_df[sites_df["direction"].isin(VALID_DIRECTIONS)].copy()

print(f"After filtering to directions {VALID_DIRECTIONS}: {len(sites_df)} sites")
print(f"\nDirection counts:")
print(sites_df["direction"].value_counts().to_string())

# Quick summary
print(f"\nSummary:")
print(f"  Total sites: {len(sites_df)}")
print(f"  Unique accessions: {sites_df['accession'].nunique()}")
print(f"  Direction counts:\n{sites_df['direction'].value_counts().to_string()}")
print(f"  Top organisms:\n{sites_df['organism'].value_counts().head(10).to_string()}")

# Save intermediate results
sites_df.to_csv(DATA_DIR / "slippage_sites_raw.csv", index=False)
print(f"\nSaved to {DATA_DIR / 'slippage_sites_raw.csv'}")

In [ ]:
def extract_window(sequence, center_pos, upstream=WINDOW_UPSTREAM, downstream=WINDOW_DOWNSTREAM):
    """
    Extract a window around a given position.
    Returns the windowed sequence and the adjusted center position,
    or None if the window extends beyond sequence boundaries.
    """
    start = center_pos - upstream
    end = center_pos + downstream

    if start < 0 or end > len(sequence):
        return None

    return sequence[start:end]


def build_positive_set(sites_df):
    """
    For each slippage site, extract the sequence window.
    Handles both forward and reverse strand features.
    """
    positives = []
    skipped = 0

    for _, row in sites_df.iterrows():
        seq = row["sequence"]

        # Handle reverse strand: reverse complement the full sequence
        # and mirror the position
        if row["strand"] == -1:
            seq = str(Seq(seq).reverse_complement())
            pos = len(seq) - row["slippage_pos"]
        else:
            pos = row["slippage_pos"]

        window = extract_window(seq, pos)
        if window is None:
            skipped += 1
            continue

        positives.append({
            "accession": row["accession"],
            "organism": row["organism"],
            "taxonomy": row["taxonomy"],
            "gene": row["gene"],
            "direction": row["direction"],
            "window_seq": window,
            "label": 1,
            "source": "genbank_slippage",
            "original_pos": row["slippage_pos"],
        })

    print(f"Extracted {len(positives)} positive windows (skipped {skipped} — too close to sequence edge)")
    return positives


positives = build_positive_set(sites_df)
print(f"\nPositive examples: {len(positives)}")

# Deduplicate by sequence
seen_seqs = set()
unique_positives = []
for p in positives:
    if p["window_seq"] not in seen_seqs:
        seen_seqs.add(p["window_seq"])
        unique_positives.append(p)

print(f"After deduplication: {len(unique_positives)} unique positive windows")
positives = unique_positives


In [ ]:
def augment_from_source(sites_df, shifts=[-15, -10, -7, -5, 5, 7, 10, 15]):
    """
    Re-extract windows with shifted centers from the original genomes.
    No padding — every nucleotide is real.
    """
    augmented = []

    for _, row in sites_df.iterrows():
        seq = row["sequence"]
        if row["strand"] == -1:
            seq = str(Seq(seq).reverse_complement())
            pos = len(seq) - row["slippage_pos"]
        else:
            pos = row["slippage_pos"]

        for shift in shifts:
            window = extract_window(seq, pos + shift)
            if window is None:
                continue
            if 'N' in window:
                continue

            augmented.append({
                "accession": row["accession"],
                "organism": row["organism"],
                "taxonomy": row.get("taxonomy", ""),
                "gene": row.get("gene", "unknown"),
                "direction": row["direction"],
                "window_seq": window,
                "label": 1,
                "source": f"augmented_shift_{shift}",
                "original_pos": row["slippage_pos"],
            })

    print(f"Generated {len(augmented)} augmented positives (no padding)")
    return augmented


augmented_positives = augment_from_source(sites_df)
positives = unique_positives + augmented_positives

print(f"\nTotal positives after augmentation: {len(positives)}")
print(f"  Original:  {sum(1 for p in positives if p['source'] == 'genbank_slippage')}")
print(f"  Augmented: {sum(1 for p in positives if 'augmented' in p['source'])}")

In [ ]:
# --- Common -1 PRF slippery site heptamer patterns ---
# Format: X XXY YYZ where the canonical motifs are:
SLIPPERY_PATTERNS = [
    # Most common -1 PRF motifs
    r'AAAAAAC', r'AAAAAAU', r'AAAAAAT',
    r'AAATTTA', r'AAATTTT', r'AAATTTC',
    r'AAAUUUA', r'AAAUUUU', r'AAAUUUC',
    r'GGGAAAC', r'GGGAAAU', r'GGGAAAT',
    r'TTTAAAC', r'UUUAAAC',
    r'TTTTTTT', r'UUUUUUU',
    r'TTTTTTA', r'UUUUUUA',
    r'TTTAAAA', r'UUUAAAA',
    r'GGGTTTT', r'GGGUUUU',
    r'CCCAAAA', r'CCCAAAG',
    r'AAAAAAT', r'AAAAAAU',
]

# Compile a unified regex (map U->T for DNA matching)
def build_slippery_regex():
    """Build regex that matches slippery heptamers in both DNA and RNA."""
    patterns_dna = set()
    for p in SLIPPERY_PATTERNS:
        # Convert to DNA
        dna = p.replace('U', 'T')
        patterns_dna.add(dna)
        # Also keep RNA version for RNA sequences
        rna = p.replace('T', 'U')
        patterns_dna.add(rna)

    # Build alternation regex
    combined = '|'.join(sorted(patterns_dna, key=len, reverse=True))
    return re.compile(combined)


slippery_re = build_slippery_regex()


def find_slippery_decoys(sequence, known_positions, min_distance=50):
    """
    Find all slippery heptamer matches in a sequence that are
    at least min_distance away from any known slippage site.
    Returns list of match positions.
    """
    decoys = []
    for match in slippery_re.finditer(sequence):
        pos = match.start() + 3  # Center of heptamer roughly
        # Check distance from all known sites
        too_close = any(abs(pos - kp) < min_distance for kp in known_positions)
        if not too_close:
            decoys.append(pos)
    return decoys


def find_cds_regions(genbank_dir):
    """
    Parse GenBank files to find all CDS regions (for random negative sampling).
    Returns dict: accession -> list of (start, end) tuples for non-slippage CDS.
    Also returns accession -> full sequence.
    """
    cds_regions = defaultdict(list)
    sequences = {}
    slippage_cds = set()
    skipped_undefined = 0
    gb_files = sorted(Path(genbank_dir).glob("*.gb"))

    for gb_file in gb_files:
        try:
            records = list(SeqIO.parse(str(gb_file), "genbank"))
        except:
            continue

        for record in records:
            acc = record.id
            try:
                sequences[acc] = str(record.seq).upper()
            except Exception:
                skipped_undefined += 1
                continue

            for feature in record.features:
                if feature.type != "CDS":
                    continue

                has_slippage = "ribosomal_slippage" in feature.qualifiers
                start = int(feature.location.start)
                end = int(feature.location.end)

                if has_slippage:
                    slippage_cds.add((acc, start, end))
                else:
                    cds_regions[acc].append((start, end))
    print(f"Skipped {skipped_undefined} undefined seqs")
    return cds_regions, sequences, slippage_cds


def generate_negatives(positives, genbank_dir, n_decoy_per_pos=NEG_RATIO_DECOY,
                       n_random_per_pos=NEG_RATIO_RANDOM):
    """
    Generate negative examples:
      Tier 1 (hard): Slippery site decoys — heptamer present but no real PRF
      Tier 2 (easy): Random CDS windows from non-frameshifted genes
    """
    print("Finding CDS regions and sequences...")
    cds_regions, sequences, slippage_cds = find_cds_regions(genbank_dir)

    # Gather known slippage positions per accession
    known_pos_by_acc = defaultdict(list)
    for p in positives:
        known_pos_by_acc[p["accession"]].append(p["original_pos"])

    n_target_decoys = len(positives) * n_decoy_per_pos
    n_target_random = len(positives) * n_random_per_pos

    # --- Tier 1: Slippery site decoys ---
    print(f"\nGenerating slippery site decoy negatives (target: {n_target_decoys})...")
    decoy_negatives = []
    all_decoy_positions = []

    for acc, seq in sequences.items():
        known = known_pos_by_acc.get(acc, [])
        decoy_positions = find_slippery_decoys(seq, known, min_distance=50)
        for pos in decoy_positions:
            all_decoy_positions.append((acc, pos, seq))

    # Randomly sample from all available decoy positions
    np.random.seed(42)
    if len(all_decoy_positions) > n_target_decoys:
        indices = np.random.choice(len(all_decoy_positions), n_target_decoys, replace=False)
        sampled_decoys = [all_decoy_positions[i] for i in indices]
    else:
        sampled_decoys = all_decoy_positions
        print(f"  Warning: only found {len(sampled_decoys)} decoy positions (target was {n_target_decoys})")

    for acc, pos, seq in sampled_decoys:
        window = extract_window(seq, pos)
        if window is None:
            continue
        decoy_negatives.append({
            "accession": acc,
            "organism": "",   # Could be filled in but not critical
            "taxonomy": "",
            "gene": "decoy",
            "direction": 0,
            "window_seq": window,
            "label": 0,
            "source": "slippery_decoy",
            "original_pos": pos,
        })

    print(f"  Generated {len(decoy_negatives)} decoy negatives")

    # --- Tier 2: Random CDS windows ---
    print(f"\nGenerating random CDS negatives (target: {n_target_random})...")
    random_negatives = []
    all_random_candidates = []

    for acc, regions in cds_regions.items():
        seq = sequences.get(acc)
        if seq is None:
            continue
        known = known_pos_by_acc.get(acc, [])

        for (start, end) in regions:
            # Only consider CDS regions long enough for our window
            low = start + WINDOW_UPSTREAM + 10
            high = end - WINDOW_DOWNSTREAM - 10
            if high <= low:
                continue
            # Sample candidate positions within this CDS
            for _ in range(3):  # Up to 3 candidates per CDS
                pos = np.random.randint(low, high)
                # Make sure it's not near a known slippage site
                too_close = any(abs(pos - kp) < 100 for kp in known)
                if not too_close:
                    all_random_candidates.append((acc, pos, seq))

    if len(all_random_candidates) > n_target_random:
        indices = np.random.choice(len(all_random_candidates), n_target_random, replace=False)
        sampled_random = [all_random_candidates[i] for i in indices]
    else:
        sampled_random = all_random_candidates
        print(f"  Warning: only found {len(sampled_random)} random positions (target was {n_target_random})")

    for acc, pos, seq in sampled_random:
        window = extract_window(seq, pos)
        if window is None:
            continue
        random_negatives.append({
            "accession": acc,
            "organism": "",
            "taxonomy": "",
            "gene": "random_cds",
            "direction": 0,
            "window_seq": window,
            "label": 0,
            "source": "random_cds",
            "original_pos": pos,
        })

    print(f"  Generated {len(random_negatives)} random CDS negatives")

    all_negatives = decoy_negatives + random_negatives
    print(f"\nTotal negatives: {len(all_negatives)}")
    return all_negatives


negatives = generate_negatives(positives, genbank_dir)

# Combine into a single dataset
all_samples = positives + negatives
print(f"\nCombined dataset: {len(all_samples)} samples")
print(f"  Positives: {sum(1 for s in all_samples if s['label'] == 1)}")
print(f"  Negatives: {sum(1 for s in all_samples if s['label'] == 0)}")

In [ ]:
# Nucleotide encoding map (T and U treated identically)
NUC_MAP = {
    'A': [1, 0, 0, 0],
    'T': [0, 1, 0, 0],
    'U': [0, 1, 0, 0],  # Same as T
    'G': [0, 0, 1, 0],
    'C': [0, 0, 0, 1],
    'N': [0.25, 0.25, 0.25, 0.25],
    'R': [0.5, 0, 0.5, 0],     # A or G
    'Y': [0, 0.5, 0, 0.5],     # T/U or C
    'S': [0, 0, 0.5, 0.5],     # G or C
    'W': [0.5, 0.5, 0, 0],     # A or T/U
    'K': [0, 0.5, 0.5, 0],     # G or T/U
    'M': [0.5, 0, 0, 0.5],     # A or C
}


def one_hot_encode(sequence):
    """
    One-hot encode a nucleotide sequence.
    Returns array of shape (seq_len, 4).
    """
    encoded = []
    for nuc in sequence.upper():
        if nuc in NUC_MAP:
            encoded.append(NUC_MAP[nuc])
        else:
            encoded.append([0.25, 0.25, 0.25, 0.25])  # Unknown
    return np.array(encoded, dtype=np.float32)


def add_codon_position(seq_len, frame_offset=0):
    """
    Generate codon position channels (3 channels, one-hot).
    frame_offset: 0, 1, or 2 depending on where the reading frame starts.

    Returns array of shape (seq_len, 3).
    """
    positions = np.zeros((seq_len, 3), dtype=np.float32)
    for i in range(seq_len):
        codon_pos = (i + frame_offset) % 3
        positions[i, codon_pos] = 1.0
    return positions


def encode_sample(sequence, frame_offset=0):
    """
    Full encoding: one-hot (4ch) + codon position (3ch) = 7 channels.
    Returns array of shape (seq_len, 7).
    """
    onehot = one_hot_encode(sequence)
    codon_pos = add_codon_position(len(sequence), frame_offset)
    return np.concatenate([onehot, codon_pos], axis=1)


def encode_dataset(samples):
    """
    Encode all samples. Returns X (features) and y (labels) arrays.
    """
    X_list = []
    y_list = []
    valid_samples = []

    for sample in samples:
        seq = sample["window_seq"]

        # Validate sequence length
        if len(seq) != WINDOW_SIZE:
            continue

        # Determine frame offset from the slippage position
        # The slippage site should be at position WINDOW_UPSTREAM in the window
        # For positives, the reading frame at the slippage site matters
        # For simplicity, we assume frame_offset=0 (slippage at a codon boundary)
        # This is a simplification — in reality you'd compute this from the CDS start
        frame_offset = WINDOW_UPSTREAM % 3

        encoded = encode_sample(seq, frame_offset)
        X_list.append(encoded)
        y_list.append(sample["label"])
        valid_samples.append(sample)

    X = np.stack(X_list, axis=0)  # (n_samples, 250, 7)
    y = np.array(y_list, dtype=np.float32)  # (n_samples,)

    print(f"Encoded {len(X)} samples → X shape: {X.shape}, y shape: {y.shape}")
    return X, y, valid_samples


X, y, valid_samples = encode_dataset(all_samples)


# Quick sanity check: visualize one encoding
print("\nSanity check — first positive sample:")
first_pos_idx = next(i for i, s in enumerate(valid_samples) if s["label"] == 1)
print(f"  Sequence (first 30 nt): {valid_samples[first_pos_idx]['window_seq'][:30]}")
print(f"  Encoding shape: {X[first_pos_idx].shape}")
print(f"  One-hot sum per position (should be ~1.0): {X[first_pos_idx, :5, :4].sum(axis=1)}")
print(f"  Codon pos sum per position (should be 1.0): {X[first_pos_idx, :5, 4:].sum(axis=1)}")


In [ ]:
all_with_augmented = valid_samples  # No further augmentation needed

print(f"Final dataset:")
print(f"  Positives: {sum(1 for s in all_with_augmented if s['label'] == 1)}")
print(f"  Negatives: {sum(1 for s in all_with_augmented if s['label'] == 0)}")

# Re-encode
X_aug, y_aug, final_samples = encode_dataset(all_with_augmented)

In [ ]:
# ============================================================
# CELL 8 — Stratified Train/Val/Test Split
# ============================================================

def stratified_split(X, y, train_frac=0.8, val_frac=0.1, test_frac=0.1, seed=42):
    """
    Split data ensuring each split has the same positive/negative ratio.
    """
    np.random.seed(seed)

    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == 0)[0]

    np.random.shuffle(pos_idx)
    np.random.shuffle(neg_idx)

    def split_indices(indices):
        n = len(indices)
        train_end = int(n * train_frac)
        val_end = int(n * (train_frac + val_frac))
        return (
            indices[:train_end],
            indices[train_end:val_end],
            indices[val_end:]
        )

    pos_train, pos_val, pos_test = split_indices(pos_idx)
    neg_train, neg_val, neg_test = split_indices(neg_idx)

    train_idx = np.concatenate([pos_train, neg_train])
    val_idx = np.concatenate([pos_val, neg_val])
    test_idx = np.concatenate([pos_test, neg_test])

    # Shuffle within each split
    np.random.shuffle(train_idx)
    np.random.shuffle(val_idx)
    np.random.shuffle(test_idx)

    return train_idx, val_idx, test_idx


print("Splitting dataset (stratified)...")
train_idx, val_idx, test_idx = stratified_split(X_aug, y_aug)

X_train, y_train = X_aug[train_idx], y_aug[train_idx]
X_val, y_val = X_aug[val_idx], y_aug[val_idx]
X_test, y_test = X_aug[test_idx], y_aug[test_idx]

print(f"\nFinal split:")
print(f"  Train: {len(X_train):>6}  ({y_train.sum():>.0f} pos / {len(y_train) - y_train.sum():>.0f} neg)")
print(f"  Val:   {len(X_val):>6}  ({y_val.sum():>.0f} pos / {len(y_val) - y_val.sum():>.0f} neg)")
print(f"  Test:  {len(X_test):>6}  ({y_test.sum():>.0f} pos / {len(y_test) - y_test.sum():>.0f} neg)")

# Verify ratios are consistent
for name, y_split in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
    ratio = (len(y_split) - y_split.sum()) / max(y_split.sum(), 1)
    print(f"  {name} neg:pos ratio = {ratio:.1f}:1")

In [ ]:
outfile = DATA_DIR / "prf_dataset.npz"

# Also save metadata as JSON for later reference
metadata = {
    "window_size": WINDOW_SIZE,
    "window_upstream": WINDOW_UPSTREAM,
    "window_downstream": WINDOW_DOWNSTREAM,
    "n_channels": 7,
    "channel_names": ["A", "T/U", "G", "C", "codon_pos_0", "codon_pos_1", "codon_pos_2"],
    "n_train": len(X_train),
    "n_val": len(X_val),
    "n_test": len(X_test),
    "n_positives_total": int(y_aug.sum()),
    "n_negatives_total": int(len(y_aug) - y_aug.sum()),
}

np.savez_compressed(
    outfile,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    X_test=X_test,
    y_test=y_test,
)

with open(DATA_DIR / "dataset_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

# Save sample-level metadata for analysis
sample_meta = pd.DataFrame([{
    "idx": i,
    "accession": s["accession"],
    "organism": s["organism"],
    "gene": s["gene"],
    "direction": s["direction"],
    "label": s["label"],
    "source": s["source"],
} for i, s in enumerate(final_samples)])
sample_meta.to_csv(DATA_DIR / "sample_metadata.csv", index=False)

print(f"\nSaved dataset to {outfile}")
print(f"  X shape: ({metadata['n_train']} + {metadata['n_val']} + {metadata['n_test']}, {WINDOW_SIZE}, 7)")
print(f"  File size: {outfile.stat().st_size / 1024 / 1024:.1f} MB")
print(f"\nMetadata saved to {DATA_DIR / 'dataset_metadata.json'}")
print(f"Sample info saved to {DATA_DIR / 'sample_metadata.csv'}")

In [ ]:
print("=" * 60)
print("DATASET VERIFICATION")
print("=" * 60)

# Reload and verify
data = np.load(outfile)
print(f"\nArrays in file: {list(data.keys())}")
print(f"X_train: {data['X_train'].shape}, dtype={data['X_train'].dtype}")
print(f"y_train: {data['y_train'].shape}, dtype={data['y_train'].dtype}")

# Verify encoding integrity
print(f"\nEncoding checks:")
print(f"  One-hot channel sums (should be ~1.0): "
      f"mean={data['X_train'][:, :, :4].sum(axis=2).mean():.4f}, "
      f"min={data['X_train'][:, :, :4].sum(axis=2).min():.4f}, "
      f"max={data['X_train'][:, :, :4].sum(axis=2).max():.4f}")
print(f"  Codon pos channel sums (should be 1.0): "
      f"mean={data['X_train'][:, :, 4:].sum(axis=2).mean():.4f}")

# Class balance
for split_name, y_split in [("Train", data["y_train"]), ("Val", data["y_val"]), ("Test", data["y_test"])]:
    n_pos = y_split.sum()
    n_neg = len(y_split) - n_pos
    ratio = n_neg / max(n_pos, 1)
    print(f"  {split_name}: {n_pos:.0f} pos, {n_neg:.0f} neg (ratio 1:{ratio:.1f})")

# Nucleotide composition check
print(f"\nNucleotide composition (train set, averaged):")
nuc_means = data["X_train"][:, :, :4].mean(axis=(0, 1))
for nuc, val in zip(["A", "T/U", "G", "C"], nuc_means):
    print(f"  {nuc}: {val:.3f}")

print(f"\n{'=' * 60}")
print(f"Pipeline complete! Your dataset is ready at: {outfile}")
print(f"Next step: build your 1D CNN and train on this data.")
print(f"{'=' * 60}")

In [ ]:
# ============================================================
# Evaluate by source — run in data.ipynb
# ============================================================

import torch
import torch.nn as nn

device = "cpu"

class SlipSlop(nn.Module):
    def __init__(self, n_channels=7, seq_len=250):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(n_channels, 32, kernel_size=7, stride=1, padding=3),
            nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(128), nn.ReLU(),
        )
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.max_pool = nn.AdaptiveMaxPool1d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
        )

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.features(x)
        avg = self.avg_pool(x)
        mx = self.max_pool(x)
        x = torch.cat([avg, mx], dim=1)
        x = self.classifier(x)
        return x

# Load trained model
model = SlipSlop().to(device)
checkpoint = torch.load("./best_prf_cnn.pt", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

# Evaluate — final_samples, X_aug, y_aug are already in memory
@torch.no_grad()
def evaluate_by_source(model, samples, X, y):
    model.eval()
    sources = sorted(set(s["source"] for s in samples))

    for source in sources:
        idx = [i for i, s in enumerate(samples) if s["source"] == source]
        if not idx:
            continue

        X_sub = torch.FloatTensor(X[idx]).to(device)
        y_sub = torch.FloatTensor(y[idx]).to(device)

        all_probs = []
        for start in range(0, len(X_sub), 256):
            logits = model(X_sub[start:start+256]).squeeze(-1)
            all_probs.append(torch.sigmoid(logits))

        probs = torch.cat(all_probs)
        preds = (probs >= 0.5).float()

        correct = (preds == y_sub).sum().item()
        acc = 100 * correct / len(y_sub)
        n_pos = (y_sub == 1).sum().item()
        n_neg = (y_sub == 0).sum().item()

        print(f"  {source:30s}  n={len(idx):>6}  pos={n_pos:>5}  neg={n_neg:>5}  "
              f"acc={acc:5.1f}%  mean_prob={probs.mean().item():.3f}")

print("Performance by sample source:\n")
evaluate_by_source(model, final_samples, X_aug, y_aug)